<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

## **Proyecto Naive Bayes**

**Autora:** Anais Aponte  
**Bootcamp:** 4Geeks Academy – Intro to Machine Learning  
**Proyecto:** Proyecto Naive Bayes  

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Instrucciones**

#### **Análisis de sentimientos**

El objetivo es crear un clasificador de reseñas de la tienda de Google Play.

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Paso 1: Carga del conjunto de datos**

El dataset utilizado en este proyecto es **playstore_reviews.csv**.

Para facilitar la reproducibilidad y evitar dependencias externas, el archivo ha sido previamente descargado y almacenado dentro del repositorio en la siguiente ruta:

📁 `data/raw/playstore_reviews.csv`

De esta forma, el notebook puede ejecutarse directamente sin necesidad de acceder a enlaces externos.

A continuación se realizan primero las importaciones necesarias y, posteriormente, la carga del dataset para iniciar el análisis.

</div>

In [1]:
# Manipulación de datos
import pandas as pd
# Preprocesamiento
from sklearn.feature_extraction.text import CountVectorizer
# División de datos
from sklearn.model_selection import train_test_split
# Modelos
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
# Métricas
from sklearn.metrics import classification_report, accuracy_score
# Importamos la librería para guardar modelos
import joblib



In [2]:
# CARGAMOS EL FICHERO CON LOS DATOS A ANALIZAR
df = pd.read_csv('../data/raw/playstore_reviews.csv')

# Visualizamos 10 registros aleatorios del dataset con sample(). Esto nos permite observar una muestra representativa de los datos,
# ya que algunos datasets pueden estar ordenados (por fecha, id, etc.).
df.sample(10)

,package_name,review,polarity
60,com.twitter.android,awsome great type of social media nothing wr...,0
881,com.rovio.angrybirds,game ruined because of ads. i felt like re-do...,0
651,com.uc.browser.en,nice.. but one request... plz include 'find t...,1
800,org.mozilla.firefox,"great app, but one problem ruins it i love th...",1
502,com.Slack,great internal comms system for our business,1
439,com.whatsapp,whatsapp i use this app now that blackberry m...,1
329,com.viber.voip,can't open after update the app crashes and d...,0
570,jabanaki.todo.todoly,doesn't work won't allow you to log in.,0
113,com.linkedin.android,groups??? ** edit: i changed my rating from o...,1
253,com.android.chrome,too many updates of too big size please only ...,0


<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación inicial**

En este dataset se identifican tres variables principales: `package_name`, `review` y `polarity`. 

La variable `review` contiene el texto del comentario y será la principal fuente de información para el modelo, ya que el objetivo es clasificar el sentimiento expresado en dicho texto. Por su parte, `polarity` actúa como variable objetivo, indicando si el comentario es negativo (0) o positivo (1).

Aunque `package_name` forma parte del dataset, no aporta información relevante para la tarea de clasificación de sentimientos, ya que el resultado depende exclusivamente del contenido del texto y no de la aplicación a la que pertenece. Por ello, esta variable será descartada en los siguientes pasos del análisis.

</div>

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

A continuación se detallan las variables incluidas en el dataset:

### 📊 Diccionario de Datos

| Variable       | Descripción                                              | Tipo        |
|----------------|----------------------------------------------------------|------------|
| package_name   | Nombre de la aplicación móvil                            | Categórico |
| review         | Comentario o reseña escrita por el usuario               | Texto      |
| polarity       | Variable objetivo (0 = comentario negativo, 1 = positivo)| Numérico   |

Este diccionario de datos será clave para interpretar correctamente las relaciones entre variables durante el análisis exploratorio (EDA).

</div>

-------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2: Estudio de variables y su contenido**

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.1: Inspección inicial del dataset**

</div>

In [3]:
# Obtener las dimensiones
df.shape

(891, 3)

In [4]:
# Obtener información sobre tipos de datos y valores no nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB


In [5]:
# Valores nulos
df.isnull().sum()

package_name    0
review          0
polarity        0
dtype: int64

In [6]:
# Vemos la proporción y numero de clases (0 y 1) en la variable objetivo
df_classes = df["polarity"].value_counts().to_frame(name="count")
df_classes["proportion"] = df["polarity"].value_counts(normalize=True)

df_classes

,count,proportion
polarity,,
0,584,0.655443
1,307,0.344557


<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación sobre la estructura del dataset**

El dataset contiene 891 registros y 3 variables. Se observa que no existen valores nulos en ninguna de las columnas, por lo que no será necesario realizar tratamiento de datos faltantes.

Las variables `package_name` y `review` son de tipo texto, mientras que `polarity` es numérica y representa la variable objetivo del modelo. 

Se observa un ligero desbalance en la variable objetivo, con mayor proporción de comentarios negativos (0 ≈ 65%) frente a positivos (1 ≈ 35%).


</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.2: Eliminar espacios y convertir a minúsculas el texto**

</div>

In [7]:
# Limpiamos el texto eliminando espacios y convirtiéndolo a minúsculas
df['review'] = df['review'].str.strip().str.lower()
df.sample(10)

,package_name,review,polarity
534,com.dropbox.android,i love dropbox...but.. i wish that the interfa...,1
478,com.Slack,"edit: gif support, yay! if this can be address...",0
888,com.rovio.angrybirds,ads are way to heavy listen to the bad reviews...,0
6,com.facebook.katana,major flaws constant updates and always gettin...,0
85,com.linkedin.android,samsung note 4 - awesome business platform! pl...,0
543,com.dropbox.android,very useful thank you android for introducing ...,1
757,com.shirantech.kantipur,please add option to provide pdf versions to d...,0
134,com.king.candycrushsaga,it sucks right now the special levels are rig...,0
850,com.hamropatro,really useful app.. đđđđ,1
194,com.imangi.templerun2,very adventurous game i love it very nice thri...,1


<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.3: Dividir el conjunto de datos en train y test: X_train, X_test, y_train, y_test**

</div>

In [8]:
# Definimos variables predictoras y objetivo (ignorando la variable `package_name` pues no aporta información relevante para la tarea de clasificación de sentimientos)
X = df["review"]
y = df["polarity"]

In [9]:
# Dividimos los datos en train y test manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)
X_train.sample(10)

776    shab i have firefox on all of my devices and i...
102    even though i am loving the new update, but th...
709    love/hate has bug and security issues. i tried...
582    evernote...forever evernote helps this busy mo...
117    app requires frequent login password why the h...
811    the best android browser fast, well designed, ...
499     perfect! very close to using the desktop client!
751    its just ok in news paper there alot of time t...
216    nyc  1 u guys should apply some same strategy ...
726    what happened to you opera... the resent updat...
Name: review, dtype: str

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.4: Transformar el texto en una matriz de recuento de palabras**

</div>

In [10]:
# Convertimos el texto en vectores numéricos mediante conteo de palabras
vec_model = CountVectorizer(stop_words="english")

X_train_vec = vec_model.fit_transform(X_train).toarray()
X_test_vec = vec_model.transform(X_test).toarray()

X_train_vec

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3272))

------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3: Construye un naive bayes (implementacion seleccionada = MultinomialNB)**

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Elección del modelo Naive Bayes**

Siguiendo la teoría del módulo, se ha optado por utilizar **MultinomialNB**, ya que este modelo está pensado para trabajar con **conteos o frecuencias**.

En este caso, las variables predictoras han sido generadas mediante CountVectorizer, lo que transforma cada texto en un vector numérico donde cada valor indica cuántas veces aparece una palabra.

Por tanto, como nuestros datos son conteos de palabras, **MultinomialNB es la opción más adecuada** según lo indicado en la teoría.

Se probarán también **GaussianNB** y **BernoulliNB** para comparar resultados, aunque en principio no parecen ser los más apropiados para este tipo de datos.

</div>

In [11]:
# Entrenamos el modelo seleccionado (MultinomialNB)
model_multi = MultinomialNB()
model_multi.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [12]:
# Realizamos predicciones
y_pred_multi = model_multi.predict(X_test_vec)

In [13]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_multi))

              precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179



<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">


### 💡 **Observación sobre el rendimiento del modelo con MultinomialNB**

Se observa que el modelo presenta un mejor rendimiento en la detección de la clase negativa (0) que en la positiva (1). Esto se refleja en un recall más elevado para la clase 0 (0.96) frente al de la clase 1 (0.66).<br><br>

Este comportamiento puede estar influenciado por el desbalanceo del dataset, donde predominan los comentarios negativos (aproximadamente 65%) frente a los positivos (35%).<br><br>

Como consecuencia, el modelo tiende a aprender mejor los patrones asociados a la clase mayoritaria, lo que facilita su detección y dificulta la identificación de la clase minoritaria.

</div>


<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3.1: Construye un naive bayes (GaussianNB)**

</div>

In [14]:
# Entrenamos el modelo (GaussianNB)
model_gauss = GaussianNB()
model_gauss.fit(X_train_vec, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [15]:
# Realizamos predicciones
y_pred_gauss = model_gauss.predict(X_test_vec)

In [16]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_gauss))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86       117
           1       0.76      0.68      0.72        62

    accuracy                           0.82       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.81      0.82      0.81       179



<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3.2: Construye un naive bayes (BernoulliNB)**

</div>

In [17]:
# Entrenamos el modelo (BernoulliNB)
model_bern = BernoulliNB()
model_bern.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [18]:
# Realizamos predicciones
y_pred_bern = model_bern.predict(X_test_vec)

In [19]:
# Evaluamos el modelo
print(classification_report(y_test, y_pred_bern))

              precision    recall  f1-score   support

           0       0.76      0.97      0.85       117
           1       0.87      0.44      0.58        62

    accuracy                           0.78       179
   macro avg       0.82      0.70      0.72       179
weighted avg       0.80      0.78      0.76       179



<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Comparación de modelos Naive Bayes**

Se han evaluado tres implementaciones del algoritmo Naive Bayes: <b>MultinomialNB</b>, <b>GaussianNB</b> y <b>BernoulliNB</b>, observándose diferencias en su rendimiento.

El modelo <b>MultinomialNB</b> obtiene el mejor resultado global, con una accuracy de aproximadamente 0.85, mostrando un equilibrio adecuado entre precision y recall en ambas clases.

Por su parte, <b>GaussianNB</b> presenta un rendimiento ligeramente inferior (accuracy ~0.82), aunque mantiene un comportamiento relativamente equilibrado entre ambas clases.

En cambio, <b>BernoulliNB</b> obtiene el peor resultado (accuracy ~0.78), destacando especialmente su bajo recall en la clase positiva (1), lo que indica que no logra detectar correctamente una parte importante de los comentarios positivos.

Estos resultados confirman lo indicado en la teoría: <b>MultinomialNB es el modelo más adecuado para trabajar con conteos de palabras</b>, como los generados mediante <code>CountVectorizer</code> en este problema de análisis de texto.

</div>

------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 4: Optimiza el modelo anterior**

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Optimización del modelo (hiperparametro alpha)**

En este paso se realiza una optimización del modelo MultinomialNB ajustando el hiperparámetro **alpha**, ya que este controla el suavizado de probabilidades y puede ayudar a mejorar el rendimiento cuando existen palabras poco frecuentes o que no aparecen en todas las clases.

No se modifica el parámetro **fit_prior**, ya que permite al modelo aprender la proporción real de clases del dataset, lo cual resulta adecuado en este caso al no existir un desbalanceo extremo.

Por tanto, se evaluará el impacto de distintos valores de alpha manteniendo el resto de parámetros por defecto.

</div>

In [20]:
# Probamos distintos valores de alpha y guardamos el mejor modelo para porder luego guardarlo
alphas = [0.1, 0.5, 1.0, 2.0]

best_acc = 0  # guardamos la mejor accuracy encontrada
best_model = None  # aquí guardaremos el mejor modelo
best_alpha = None  # guardamos el mejor alpha

for a in alphas:
    
    # Entrenamos modelo con el alpha actual
    model = MultinomialNB(alpha=a)
    model.fit(X_train_vec, y_train)
    
    # Predecimos
    y_pred = model.predict(X_test_vec)
    
    # Calculamos accuracy
    acc = accuracy_score(y_test, y_pred)
    print(f"alpha={a} -> accuracy={acc:.4f}")
    
    # Si es mejor que el anterior, lo guardamos
    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_alpha = a

# Mostramos el mejor resultado
print(f"\nMejor alpha: {best_alpha} con accuracy: {best_acc:.4f}")

alpha=0.1 -> accuracy=0.8547
alpha=0.5 -> accuracy=0.8715
alpha=1.0 -> accuracy=0.8547
alpha=2.0 -> accuracy=0.8268

Mejor alpha: 0.5 con accuracy: 0.8715


<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación de la optimizacion con alpha**

Despues de evaluar distintos valores del hiperparámetro **alpha** en el modelo MultinomialNB.

Se observa que el mejor rendimiento se obtiene con **alpha=0.5**, alcanzando una accuracy de aproximadamente 0.87, mientras que valores más altos como 2.0 reducen el rendimiento.

Por tanto, se selecciona **alpha=0.5** como la configuración óptima para este modelo.

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Optimización del modelo (Random Forest)**

Se prueba un modelo Random Forest como alternativa, siguiendo los requisitos del proyecto, con el objetivo de evaluar si es posible mejorar el rendimiento obtenido con MultinomialNB.

</div>

In [21]:
# Entrenamos un modelo ahora con Random Forest
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_vec, y_train)

# Realizamos predicciones sobre el conjunto de test
y_pred_rf = model_rf.predict(X_test_vec)

# Calculamos el accuracy del modelo
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy Random Forest: {acc_rf:.4f}")

Accuracy Random Forest: 0.8212


<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación la optimización con Random Forest**

Se ha probado un modelo Random Forest como alternativa para intentar mejorar los resultados obtenidos con MultinomialNB.

Sin embargo, el accuracy alcanzado por Random Forest (~0.82) es inferior al obtenido por MultinomialNB con los mejores valores de alpha, lo que confirma que Naive Bayes sigue siendo la opción más adecuada para este problema de clasificación de texto.

Esto es coherente con la teoría, ya que MultinomialNB está específicamente diseñado para trabajar con conteos de palabras, mientras que Random Forest no es un modelo especializado en este tipo de datos.

</div>

------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 5: Guardar el modelo**

</div>

In [22]:
# Guardamos el mejor modelo entrenado
joblib.dump(best_model, "../models/multinomial_nb_best.sav")

# Guardamos también el vectorizador
joblib.dump(vec_model, "../models/vectorizer.sav")

['../models/vectorizer.sav']

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación sobre el guardado del modelo**

Se utiliza la librería joblib para almacenar el modelo entrenado, ya que es más eficiente para trabajar con estructuras de datos grandes como las matrices generadas en NLP.

Además, se guarda también el vectorizador (CountVectorizer), ya que es necesario para transformar nuevos textos antes de realizar predicciones con el modelo.

</div>

------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 6: Explora otras alternativas**

</div>

<div style="background-color:#fff3cd; border-left:6px solid #f1c40f; padding:15px; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación sobre el uso de Boosting como alternativa**

Aunque Random Forest no mostró mejoras frente a Naive Bayes, se decide probar Boosting como alternativa, ya que introduce un enfoque diferente basado en aprendizaje secuencial donde cada modelo corrige los errores del anterior.

Sin embargo, al tratarse de datos de texto representados como vectores de alta dimensionalidad, este tipo de modelos basados en árboles no suelen ser los más adecuados, lo que podría limitar su rendimiento frente a modelos específicos para NLP como MultinomialNB.

</div>

In [23]:
# Entrenamos un modelo con Gradient Boosting
model_boost = GradientBoostingClassifier(random_state=42)
model_boost.fit(X_train_vec, y_train)

# Realizamos predicciones sobre el conjunto de test
y_pred_boost = model_boost.predict(X_test_vec)

# Calculamos el accuracy del modelo
acc_boost = accuracy_score(y_test, y_pred_boost)
print(f"Accuracy Gradient Boosting: {acc_boost:.4f}")

Accuracy Gradient Boosting: 0.7542


<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

### 💡 **Conclusiónn final**

En este proyecto se ha abordado un problema de clasificación de texto utilizando distintas aproximaciones de Machine Learning.

En primer lugar, se ha aplicado MultinomialNB, modelo especialmente diseñado para trabajar con datos de tipo texto representados como conteos de palabras. Este modelo ha demostrado el mejor rendimiento, alcanzando una accuracy cercana a 0.87 tras la optimización del hiperparámetro alpha.

Posteriormente, se han explorado modelos alternativos como GaussianNB, BernoulliNB, Random Forest y Gradient Boosting, observándose que ninguno de ellos logra superar el rendimiento de MultinomialNB. En particular, los modelos basados en árboles presentan un rendimiento inferior, lo que confirma que no son la opción más adecuada para datos de texto en formato de alta dimensionalidad.

Estos resultados son coherentes con la teoría, ya que Naive Bayes, y en concreto MultinomialNB, está específicamente diseñado para problemas de clasificación de texto, donde la representación mediante conteo de palabras encaja perfectamente con sus supuestos.

Por tanto, se concluye que MultinomialNB es el modelo más adecuado para este problema, destacando además por su simplicidad, eficiencia y buen rendimiento.
</div>